# Murphy 2 · Chapter 36 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#A36). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## A36.1 · Causal inference: backdoor adjustment & IPW under confounding  
*Section §36.4 · code*

Simulate N = 20,000 patients where severity Z ~ Bernoulli(0.5) drives both treatment assignment P(X=1 | Z) = 0.15 + 0.70 Z and outcome Y = 0.80 + 0.12 X − 0.50 Z + ε (true causal ATE = +0.12). Compute (a) the naive observational difference E[Y|X=1] − E[Y|X=0], (b) Pearl's backdoor adjustment across Z ∈ {0, 1}, and (c) the Inverse Probability Weighting (IPW) estimate.

<details><summary><b>Hint</b></summary>

For IPW, weight each unit by X/e(Z) − (1−X)/(1−e(Z)) where e(Z) = P(X=1 | Z).

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(36)
N = 20_000
Z = rng.binomial(1, 0.5, N)
e_z = ...  # TODO: propensity score P(X=1 | Z)
X = rng.binomial(1, e_z)
Y = 0.80 + 0.12 * X - 0.50 * Z + rng.normal(0, 0.05, N)
obs_diff = Y[X == 1].mean() - Y[X == 0].mean()
backdoor = sum(np.mean(Z == z) * (Y[(X == 1) & (Z == z)].mean() - Y[(X == 0) & (Z == z)].mean()) for z in [0, 1])
ipw = ...  # TODO: Horvitz-Thompson IPW estimator
print(f"naive observational E[Y|X=1] - E[Y|X=0]: {obs_diff:+.4f}  (confounded!)")
print(f"Pearl backdoor adjustment ATE:           {backdoor:+.4f}  (true +0.1200)")
print(f"Inverse Probability Weighting (IPW) ATE: {ipw:+.4f}  (true +0.1200)")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(36)
N = 20_000
Z = rng.binomial(1, 0.5, N)
e_z = np.where(Z == 1, 0.85, 0.15)  # propensity score P(X=1 | Z)
X = rng.binomial(1, e_z)
Y = 0.80 + 0.12 * X - 0.50 * Z + rng.normal(0, 0.05, N)
obs_diff = Y[X == 1].mean() - Y[X == 0].mean()
backdoor = sum(np.mean(Z == z) * (Y[(X == 1) & (Z == z)].mean() - Y[(X == 0) & (Z == z)].mean()) for z in [0, 1])
ipw = np.mean(X * Y / e_z - (1 - X) * Y / (1 - e_z))  # Horvitz-Thompson IPW estimator
print(f"naive observational E[Y|X=1] - E[Y|X=0]: {obs_diff:+.4f}  (confounded!)")
print(f"Pearl backdoor adjustment ATE:           {backdoor:+.4f}  (true +0.1200)")
print(f"Inverse Probability Weighting (IPW) ATE: {ipw:+.4f}  (true +0.1200)")

<details><summary><b>Answer and explanation</b></summary>

Naive observational conditioning gives −0.16 (making a beneficial +0.12 treatment look harmful because severe patients take it 5.7× as often!). Both Pearl's backdoor adjustment and propensity-score IPW erase the confounding bias and recover the true +0.12 causal effect (Murphy 2 §36.4).

Expected output:
```
naive observational E[Y|X=1] - E[Y|X=0]: -0.2344  (confounded!)
Pearl backdoor adjustment ATE:           +0.1201  (true +0.1200)
Inverse Probability Weighting (IPW) ATE: +0.1025  (true +0.1200)
```

</details>